# Train SciBERT on ZO_up

### Prepare environment

In [1]:
import os

# needs to be executed before importing torch or transformers
# server specific: only use last 3 gpus (on rattle.ifi.uzh.ch)
# os.environ["CUDA_VISIBLE_DEVICES"] = "2,3,4"

# set the home directory for huggingface transformers (where the models are saved)
# by default this is '~/.cache/huggingface/hub'
# see https://stackoverflow.com/questions/61798573/where-does-hugging-faces-transformers-save-models
# server specific:
# os.environ["HF_HOME"] = "/srv/scratch2/dbielik/.cache/huggingface"

import torch
from pathlib import Path
from transformers import set_seed

# increase the number of elements printed in the tensor
torch.set_printoptions(threshold=10_000)

# set the number of threads for BLAS libraries (for maxmizing reproducibility)
# warning: this will slow down the training
USE_DETERMINISTIC_ALGORITHMS = False
torch.use_deterministic_algorithms(USE_DETERMINISTIC_ALGORITHMS)
if USE_DETERMINISTIC_ALGORITHMS:
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"

# check if CUDA is available
if not torch.cuda.is_available():
    print("Warning: CUDA not available!")

# path of the directory containing this file
BASE_DIR_PATH = Path.cwd().parent
# path of the data directory
DATA_DIR_PATH = BASE_DIR_PATH / "experiments" / "data"

HOME_DIR = os.path.expanduser("~")
CHECKPOINT_PATH = (
    os.getenv("HF_HOME") or HOME_DIR + "/.cache/huggingface"
) + "/checkpoints"

# set the seed for reproducibility
SEED = 1337
set_seed(SEED)


In [2]:
from enum import Enum
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
import pickle


class DatasetType(Enum):
    """Enum for the dataset type."""

    """Zora + OSDG upsampled dataset"""
    ZO_UP = "zo_up"
    """SwissText Shared Task 1 dataset (Zurich NLP)"""
    SWISSTEXT_SHARED_TASK1 = "swisstext_shared_task1"


TEST_SIZE = 0.3
DATASET_TYPE = DatasetType.ZO_UP

# load the dataset
# note: if you don't have the data in the folder, use the download-data.sh script

match DATASET_TYPE:
    case DatasetType.ZO_UP:
        # dont need to use manual features as class_encode_column will create ClassLabel
        # careful: watch out for the order of the ClassLabel as it doesn't map directly to the SDG class. need use mapping functions (id2label, label2id)
        # sdgs = [str(i) for i in range(1, 18)] + ["non-relevant"]
        # features = Features({"sdg": ClassLabel(num_classes=len(sdgs), names=sdgs), "abstract": Value("string")})

        dataset = load_dataset("csv", data_files=str(DATA_DIR_PATH / "zo_up.csv"))
        dataset = dataset.rename_columns({"sdg": "SDG", "abstract": "ABSTRACT"})

        def convert_sdg_to_0indexed_int(d):
            d["SDG"] = int(d["SDG"]) - 1
            return d

        dataset = dataset.map(convert_sdg_to_0indexed_int)

        train_df = dataset["train"].to_pandas()
        train_data, test_data = train_test_split(
            train_df, test_size=TEST_SIZE, stratify=train_df["SDG"], random_state=SEED
        )

        dataset["train"] = Dataset.from_pandas(train_data)
        dataset["test"] = Dataset.from_pandas(test_data)

    case DatasetType.SWISSTEXT_SHARED_TASK1:
        dataset = load_dataset(
            "json",
            data_files=str(
                DATA_DIR_PATH / "swisstext-2024-sharedtask" / "task1-train.jsonl"
            ),
        )
        dataset = dataset["train"].train_test_split(test_size=TEST_SIZE, seed=SEED)

print(dataset["train"].features)
example = dataset["train"][0]
print("Example instance:\t", example)


# Label encodings / mappings
labels = set(dataset["train"]["SDG"])

# Create id2label and label2id dictionaries
id2label = {i: str(i + 1) for i in range(len(labels))}
label2id = {str(i + 1): i for i in range(len(labels))}

# Print the results
print("id2label:", id2label)
print("label2id:", label2id)

# save the encodings to a file for later use
ENCODING_DIR = BASE_DIR_PATH / "encodings" / DATASET_TYPE.value
# create the directory if it doesn't exist
ENCODING_DIR.mkdir(parents=True, exist_ok=True)
with open(ENCODING_DIR / "id2label_natural.pkl", "wb") as f:
    pickle.dump(id2label, f)

with open(ENCODING_DIR / "label2id_natural.pkl", "wb") as f:
    pickle.dump(label2id, f)

# verify that encodings work properly on example instance
# example instance has label 9 in the csv file
# example instance has label 16 in the encoded dataset
assert example["SDG"] == label2id[id2label[example["SDG"]]]
print("Encoded (label2id) label:\t", example["SDG"])
print("Decoded (id2label) label:\t", id2label[example["SDG"]])

print(id2label[16], label2id[id2label[16]], label2id["9"])


# whether the text should be lowered or not
SHOULD_LOWER = False


def preprocess_data(
    tokenizer, padding="max_length", max_length=512, include_labels=True
):
    def _preprocess_data(instances):
        match DATASET_TYPE:
            case DatasetType.SWISSTEXT_SHARED_TASK1:
                # take a batch of titles and abstracts and concat them
                titles = instances["TITLE"]
                abstracts = instances["ABSTRACT"]
                texts = [
                    f"{title} {abstract}" for title, abstract in zip(titles, abstracts)
                ]
            case DatasetType.ZO_UP:
                texts = instances["ABSTRACT"]

        if SHOULD_LOWER:
            texts = [text.lower() for text in texts]

        # encode
        encoding = tokenizer(
            texts,
            padding=padding,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )

        if include_labels:
            # add labels
            encoding["label"] = torch.tensor([label for label in instances["SDG"]])

        return encoding

    return _preprocess_data


{'SDG': Value(dtype='int64', id=None), 'ABSTRACT': Value(dtype='string', id=None), 'id': Value(dtype='string', id=None), 'sdg_desc_short': Value(dtype='string', id=None), 'sdg_desc_long': Value(dtype='string', id=None), '__index_level_0__': Value(dtype='int64', id=None)}
Example instance:	 {'SDG': 8, 'ABSTRACT': 'The scheme gives enterprises with business activity in Norway a tax credit on their R&D projects. The R&D content must be approved by the Research Council of Norway ex ante. In 2009, the cap on expenses per enterprise for intramural R&D projects increased to NOK 5.5 million (previously it was N0K 4 million), and NOK11 million (previously it was NOK 8 million) for projects conducted at an R&D institution.', 'id': None, 'sdg_desc_short': None, 'sdg_desc_long': None, '__index_level_0__': 492}
id2label: {0: '1', 1: '2', 2: '3', 3: '4', 4: '5', 5: '6', 6: '7', 7: '8', 8: '9', 9: '10', 10: '11', 11: '12', 12: '13', 13: '14', 14: '15', 15: '16', 16: '17'}
label2id: {'1': 0, '2': 1, '

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

HF_MODEL_NAME = "allenai/scibert_scivocab_cased"
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_NAME, do_lower_case=False)
model = AutoModelForSequenceClassification.from_pretrained(
    HF_MODEL_NAME,
    id2label=id2label,
    label2id=label2id
).to("cuda")

encoded_dataset = dataset.map(
    preprocess_data(tokenizer), batched=True, remove_columns=dataset["train"].column_names
)
encoded_dataset.set_format("torch")

/home/dvdblk/miniconda3/envs/thesis_25/lib/python3.12/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at allenai/scibert_scivocab_cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/home/dvdblk/miniconda3/envs/thesis_25/lib/python3.12/site-packages/dill/_dill.py:414: PicklingWarning: Cannot locate reference to <enum 'DatasetType'>.
  StockPickler.save(self, obj, save_persistent_id)
/home/dvdblk/miniconda3/envs/thesis_25/lib/python3.12/site-packages/dill/_dill.py:414: PicklingWarning: Cannot pickle <enum 'DatasetType'>: __main__.DatasetType has recursive self

Map:   0%|          | 0/630 [00:00<?, ? examples/s]

Map:   0%|          | 0/271 [00:00<?, ? examples/s]

In [4]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import TrainingArguments, Trainer, EvalPrediction

BATCH_SIZE = 8
METRIC_NAME = "accuracy"

args = TrainingArguments(
    f"{CHECKPOINT_PATH}/sdg-scibert/{HF_MODEL_NAME}-{DATASET_TYPE.value}",
    evaluation_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=20,
    load_best_model_at_end=True,
    save_total_limit=2,
    metric_for_best_model=METRIC_NAME,
    seed=SEED,
    report_to="none",
    run_name=f"{HF_MODEL_NAME}-{DATASET_TYPE.value}"
)

def compute_metrics(pred: EvalPrediction):
    labels = pred.label_ids
    accuracy = accuracy_score(labels, pred.predictions.argmax(-1))
    precision, recall, f1, _ = precision_recall_fscore_support(labels, pred.predictions.argmax(-1), average="weighted")
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

trainer = Trainer(
    model,
    args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)


/home/dvdblk/miniconda3/envs/thesis_25/lib/python3.12/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [5]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,2.598300,2.073226,0.494465,0.565876,0.494465,0.428780
2,1.579900,1.236395,0.741697,0.754450,0.741697,0.732851
3,0.843400,1.019641,0.719557,0.721308,0.719557,0.711365
4,0.455300,1.020564,0.745387,0.750757,0.745387,0.739337
5,0.243800,0.997108,0.749077,0.744938,0.749077,0.742176
6,0.124900,1.028452,0.745387,0.738835,0.745387,0.738071
7,0.066500,1.108102,0.771218,0.771331,0.771218,0.763737
8,0.039800,1.122713,0.767528,0.768832,0.767528,0.761738
9,0.025200,1.166452,0.760148,0.771486,0.760148,0.756475
10,0.017100,1.200034,0.760148,0.777389,0.760148,0.758944


/home/dvdblk/miniconda3/envs/thesis_25/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/dvdblk/miniconda3/envs/thesis_25/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/dvdblk/miniconda3/envs/thesis_25/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metr

TrainOutput(global_step=1580, training_loss=0.3026727230677122, metrics={'train_runtime': 392.694, 'train_samples_per_second': 32.086, 'train_steps_per_second': 4.023, 'total_flos': 3315645785088000.0, 'train_loss': 0.3026727230677122, 'epoch': 20.0})

In [6]:
trainer.evaluate()

/home/dvdblk/miniconda3/envs/thesis_25/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


{'eval_loss': 1.1081024408340454,
 'eval_accuracy': 0.7712177121771218,
 'eval_precision': 0.7713308491857724,
 'eval_recall': 0.7712177121771218,
 'eval_f1': 0.7637370478203962,
 'eval_runtime': 2.0098,
 'eval_samples_per_second': 134.839,
 'eval_steps_per_second': 16.917,
 'epoch': 20.0}

In [7]:
from sklearn.metrics import classification_report

model.eval()

# manual evaluation to show classifcation_report
true_labels = []
logits = []

for batch in encoded_dataset["test"]:
    batch = {k: v.to(trainer.args.device).unsqueeze(0) for k, v in batch.items()}
    label = batch.pop("label")

    # Forward pass
    with torch.no_grad():
        out = model(**batch)

    true_labels.append(label.item())
    logits.extend(out.logits.tolist())

probabilites = torch.nn.functional.softmax(torch.tensor(logits), dim=-1)
pred_labels = torch.argmax(probabilites, dim=-1).tolist()

report = classification_report(true_labels, pred_labels, target_names=[f"SDG {id2label[i]}" for i in range(len(labels))], digits=4)

/home/dvdblk/miniconda3/envs/thesis_25/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/dvdblk/miniconda3/envs/thesis_25/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/dvdblk/miniconda3/envs/thesis_25/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metr

In [8]:
print(report)

              precision    recall  f1-score   support

       SDG 1     0.8333    0.8824    0.8571        17
       SDG 2     0.7619    0.9412    0.8421        17
       SDG 3     0.8235    0.8235    0.8235        17
       SDG 4     0.9286    0.7647    0.8387        17
       SDG 5     0.6842    0.7647    0.7222        17
       SDG 6     0.8000    1.0000    0.8889        16
       SDG 7     0.7333    0.6875    0.7097        16
       SDG 8     0.7273    0.4706    0.5714        17
       SDG 9     0.6500    0.7647    0.7027        17
      SDG 10     0.6250    0.5882    0.6061        17
      SDG 11     0.8889    0.9412    0.9143        17
      SDG 12     0.6154    0.4706    0.5333        17
      SDG 13     0.8421    0.9412    0.8889        17
      SDG 14     0.9444    1.0000    0.9714        17
      SDG 15     0.9091    0.5882    0.7143        17
      SDG 16     0.6190    0.7647    0.6842        17
      SDG 17     0.0000    0.0000    0.0000         1

    accuracy              

In [9]:
print(model)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(31116, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [10]:
import transformers
pred = transformers.pipeline(
    "text-classification",
    model=model,
    batch_size=8,
    tokenizer=tokenizer,
    truncation=True,
    padding=True,
    max_length=512,
    device=0,
    top_k=None,     # equal to return_all_scores=True
)

sample_sentence = """Ensure access to affordable, reliable, sustainable and modern energy for all """
pred([sample_sentence])

[[{'label': '7', 'score': 0.9729177951812744},
  {'label': '12', 'score': 0.008652706630527973},
  {'label': '9', 'score': 0.0035178083926439285},
  {'label': '6', 'score': 0.0018268582643941045},
  {'label': '13', 'score': 0.0017908847657963634},
  {'label': '8', 'score': 0.0012358594685792923},
  {'label': '2', 'score': 0.0011122639989480376},
  {'label': '14', 'score': 0.001051721628755331},
  {'label': '4', 'score': 0.0010491475695744157},
  {'label': '10', 'score': 0.0010272173676639795},
  {'label': '5', 'score': 0.000987328472547233},
  {'label': '3', 'score': 0.0009558788151480258},
  {'label': '1', 'score': 0.0008312798454426229},
  {'label': '16', 'score': 0.0008084650617092848},
  {'label': '11', 'score': 0.0008029652526602149},
  {'label': '15', 'score': 0.000758261710871011},
  {'label': '17', 'score': 0.0006735210190527141}]]